In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [2]:
# Load the imdb dataset
max_features = 10000  # Vocabulary size
(X_train,y_train),(X_test,y_test) = imdb.load_data(num_words=max_features)

# Print the shape of the data
print(f'Training data shape: {X_train.shape}, Training labels shape: {y_train.shape}')
print(f'Testing data shape: {X_test.shape}, Testing labels shape: {y_test.shape}')


17464789/17464789 [==============================] - 3s 0us/step
Training data shape: (25000,), Training labels shape: (25000,)
Testing data shape: (25000,), Testing labels shape: (25000,)


In [4]:
# Let;s inspect the datasets
sample_review = X_train[0]
sample_label = y_train[0]

print(f'Sample review (encoded): {sample_review}')
print(f'Sample review length: {len(sample_review)} words')



Sample review (encoded): [1, 14, 22, 16, 43, 530, 973, 1622, 1385, 65, 458, 4468, 66, 3941, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 4536, 1111, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 2025, 19, 14, 22, 4, 1920, 4613, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 1247, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 2223, 5244, 16, 480, 66, 3785, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 1415, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 5952, 15, 256, 4, 2, 7, 3766, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 1029, 13, 104, 88, 4, 381, 15, 297, 98, 32, 2071, 56, 26, 141, 6, 194, 7486, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 5535, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 1334, 88, 12, 16, 283, 5, 16, 4472, 113, 103, 32, 15, 16, 5345, 19, 178, 32]
Sample review length: 218 words


In [ ]:
# Mapping of word indices to actual words
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}


# Decode the sample review back to words
decoded_review = ' '.join([reverse_word_index.get(i - 3, '?') for i in sample_review])
print(f'Decoded review: {decoded_review}')

Decoded review: ? this film was just brilliant casting location scenery story direction everyone's really suited the part they played and you could just imagine being there robert ? is an amazing actor and now the same being director ? father came from the same scottish island as myself so i loved the fact there was a real connection with this film the witty remarks throughout the film were great it was just brilliant so much that i bought the film as soon as it was released for ? and would recommend it to everyone to watch and the fly fishing was amazing really cried at the end it was so sad and you know what they say if you cry at a film it must have been good and this definitely was also ? to the two little boy's that played the ? of norman and paul they were just brilliant children are often left out of the ? list i think because the stars that play them all grown up are such a big profile for the whole film but these children are amazing and should be praised for what they have do

In [9]:
# Let's pre-pad using sequence padding
maxlen = 500  # Maximum length of reviews
X_train = sequence.pad_sequences(X_train, maxlen=maxlen)
X_test = sequence.pad_sequences(X_test, maxlen=maxlen)

X_train

array([[   0,    0,    0, ...,   19,  178,   32],
       [   0,    0,    0, ...,   16,  145,   95],
       [   0,    0,    0, ...,    7,  129,  113],
       ...,
       [   0,    0,    0, ...,    4, 3586,    2],
       [   0,    0,    0, ...,   12,    9,   23],
       [   0,    0,    0, ...,  204,  131,    9]], dtype=int32)

In [11]:
# Train Simple RNN model

model = Sequential()
model.add(Embedding(input_dim=max_features, output_dim=128, input_length=maxlen)) # Embedding layer i.e. converting words to vectors
model.add(SimpleRNN(128))  # Simple RNN layer with 128 hidden units/layers
model.add(Dense(1, activation='sigmoid'))  # Output layer for binary classification

In [12]:
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_1 (Embedding)     (None, 500, 128)          1280000   
                                                                 
 simple_rnn (SimpleRNN)      (None, 128)               32896     
                                                                 
 dense (Dense)               (None, 1)                 129       
                                                                 
Total params: 1313025 (5.01 MB)
Trainable params: 1313025 (5.01 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [14]:
# Create Early Stopping call back
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [15]:
# Train the model with early stopping
model.fit(X_train, y_train,
          epochs=10,
          batch_size=32,
          validation_split=0.2,
          callbacks=[early_stopping])

# Evaluate the model
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f'Test Loss: {test_loss}, Test Accuracy: {test_acc}')

Epoch 1/10
625/625 [==============================] - 49s 78ms/step - loss: 0.6330 - accuracy: 0.6241 - val_loss: 0.6382 - val_accuracy: 0.6476
Epoch 2/10
625/625 [==============================] - 49s 79ms/step - loss: 0.5511 - accuracy: 0.7200 - val_loss: 0.6375 - val_accuracy: 0.6196
Epoch 3/10
625/625 [==============================] - 49s 79ms/step - loss: 0.5517 - accuracy: 0.7193 - val_loss: 0.6245 - val_accuracy: 0.6530
Epoch 4/10
625/625 [==============================] - 49s 78ms/step - loss: 0.4658 - accuracy: 0.7745 - val_loss: 0.6179 - val_accuracy: 0.6956
Epoch 5/10
625/625 [==============================] - 49s 78ms/step - loss: 0.3647 - accuracy: 0.8434 - val_loss: 0.5333 - val_accuracy: 0.7554
Epoch 6/10
625/625 [==============================] - 49s 79ms/step - loss: 0.2740 - accuracy: 0.8928 - val_loss: 0.5755 - val_accuracy: 0.7456
Epoch 7/10
782/782 [==============================] - 15s 19ms/step - loss: 0.5248 - accuracy: 0.7638
Test Loss: 0.5247649550437927, Tes

In [16]:
# Save the model
model.save('simplernn_imdb_model.h5')

/Users/satyakibasu/Documents/Satyaki/python_code/gen-ai/p311env/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
